<a href="https://colab.research.google.com/github/IrisCheon/NLP-practice/blob/main/Article%E2%80%93Summary_Relationship_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exploring How News Articles Are Condensed into Summaries

This project explores the relationship between full news articles and their summaries using the CNN/DailyMail dataset. Rather than focusing on classification or model performance, the analysis examines how summaries compress information, reuse article vocabulary, and rely on different parts of the original article text.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

In [2]:
from datasets import load_dataset

In [3]:
dataset = load_dataset("cnn_dailymail", "3.0.0")

README.md: 0.00B [00:00, ?B/s]

3.0.0/train-00000-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00002-of-00003.parquet:   0%|          | 0.00/259M [00:00<?, ?B/s]

3.0.0/validation-00000-of-00001.parquet:   0%|          | 0.00/34.7M [00:00<?, ?B/s]

3.0.0/test-00000-of-00001.parquet:   0%|          | 0.00/30.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

#■ 데이터 확인/전처리

In [10]:
dataset.keys()

dict_keys(['train', 'validation', 'test'])

In [12]:
train_df = pd.DataFrame(dataset["train"])

In [13]:
train_df.head()

,article,highlights,id
0,"LONDON, England (Reuters) -- Harry Potter star...",Harry Potter star Daniel Radcliffe gets £20M f...,42c027e4ff9730fbb3de84c1af0d2c506e41c3e4
1,Editor's note: In our Behind the Scenes series...,Mentally ill inmates in Miami are housed on th...,ee8871b15c50d0db17b0179a6d2beab35065f1e9
2,"MINNEAPOLIS, Minnesota (CNN) -- Drivers who we...","NEW: ""I thought I was going to die,"" driver sa...",06352019a19ae31e527f37f7571c6dd7f0c5da37
3,WASHINGTON (CNN) -- Doctors removed five small...,"Five small polyps found during procedure; ""non...",24521a2abb2e1f5e34e6824e0f9e56904a2b0e88
4,(CNN) -- The National Football League has ind...,"NEW: NFL chief, Atlanta Falcons owner critical...",7fe70cc8b12fab2d0a258fababf7d9c6b5e1262a


In [ ]:
train_df = train_df.drop("id", axis=1)

In [16]:
train_df

,article,highlights
0,"LONDON, England (Reuters) -- Harry Potter star...",Harry Potter star Daniel Radcliffe gets £20M f...
1,Editor's note: In our Behind the Scenes series...,Mentally ill inmates in Miami are housed on th...
2,"MINNEAPOLIS, Minnesota (CNN) -- Drivers who we...","NEW: ""I thought I was going to die,"" driver sa..."
3,WASHINGTON (CNN) -- Doctors removed five small...,"Five small polyps found during procedure; ""non..."
4,(CNN) -- The National Football League has ind...,"NEW: NFL chief, Atlanta Falcons owner critical..."
...,...,...
287108,"The nine-year-old daughter of a black, unarmed...","Rumain Brisbon, 34, was killed after Phoenix p..."
287109,Legalising assisted suicide is a slippery slop...,"Theo Boer, a European assisted suicide watchdo..."
287110,A group calling itself 'The Women of the 99 Pe...,Ohio congressman criticised for 'condoning the...
287111,Most men enjoy a good pint of lager or real al...,The Black Country Ale Tairsters have been to 1...


In [17]:
sample_df = train_df.sample(5000, random_state = 42)
sample_df

,article,highlights
272581,Nasa has warned of an impending asteroid pass ...,2004 BL86 will pass about three times the dist...
772,"BAGHDAD, Iraq (CNN) -- Iraq's most powerful Su...","Iraqi Islamic Party calls Quran incident ""blat..."
171868,By . David Kent . Andy Carroll has taken an un...,Carroll takes to Instagram to post selfie ahea...
63167,Los Angeles (CNN) -- Los Angeles has long been...,Pop stars from all over Europe are setting the...
68522,London (CNN) -- Few shows can claim such an au...,NEW: Young athletes light the Olympic cauldron...
...,...,...
271171,A cheat was caught by his blonde fiancée when ...,"Deron Yapp, 24, was caught cheating by his blo..."
146080,By . Daily Mail Reporter . PUBLISHED: . 16:51 ...,Cancer mortality rates have declined continual...
270020,"Ched Evans, pictured outside his home with his...",League One football club Oldham Athletic are s...
126659,"By . Tamara Cohen, Political Reporter . PUBLIS...",MPs say safeguarding Britain's Olympic legacy ...


## ■ article / summary 길이 분석

In [18]:
sample_df["article_word_count"] = (sample_df["article"].str.split().str.len())
sample_df["summary_word_count"] = (sample_df["highlights"].str.split().str.len())

sample_df.head()

,article,highlights,article_word_count,summary_word_count
272581,Nasa has warned of an impending asteroid pass ...,2004 BL86 will pass about three times the dist...,524,43
772,"BAGHDAD, Iraq (CNN) -- Iraq's most powerful Su...","Iraqi Islamic Party calls Quran incident ""blat...",673,46
171868,By . David Kent . Andy Carroll has taken an un...,Carroll takes to Instagram to post selfie ahea...,568,37
63167,Los Angeles (CNN) -- Los Angeles has long been...,Pop stars from all over Europe are setting the...,1026,43
68522,London (CNN) -- Few shows can claim such an au...,NEW: Young athletes light the Olympic cauldron...,805,47


##■ article / summary 요약률 분석

In [19]:
sample_df["compression_ratio"] = (sample_df["summary_word_count"] / sample_df["article_word_count"])

sample_df["compression_ratio"].mean()

np.float64(0.09000260344186266)

※ highlights 단어 수는 article 단어 수의 평균 9%

##■ article / summary overlap

In [20]:
def word_overlap(article, summary):
    article_words = set(article.lower().split())    # set으로 바꾸기
    summary_words = set(summary.lower().split())

    overlap = article_words.intersection(summary_words)
            # 교집합 구하기(intersection)
    return len(overlap) / len(summary_words)

In [21]:
sample_df["word_overlap"] = sample_df.apply(
    lambda row: word_overlap(
        row["article"],
        row["highlights"]
    ), axis=1
)

sample_df.head()

,article,highlights,article_word_count,summary_word_count,compression_ratio,word_overlap
272581,Nasa has warned of an impending asteroid pass ...,2004 BL86 will pass about three times the dist...,524,43,0.082061,0.911765
772,"BAGHDAD, Iraq (CNN) -- Iraq's most powerful Su...","Iraqi Islamic Party calls Quran incident ""blat...",673,46,0.068351,0.815789
171868,By . David Kent . Andy Carroll has taken an un...,Carroll takes to Instagram to post selfie ahea...,568,37,0.065141,0.843750
63167,Los Angeles (CNN) -- Los Angeles has long been...,Pop stars from all over Europe are setting the...,1026,43,0.041910,0.857143
68522,London (CNN) -- Few shows can claim such an au...,NEW: Young athletes light the Olympic cauldron...,805,47,0.058385,0.611111


In [22]:
sample_df["word_overlap"].mean()

np.float64(0.7985356331537298)

※ highlights의 단어 중 약 80%가량이 article 본문에 있던 단어
- extractive summary에 가까움

#■ TF-IDF keyword

In [23]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [24]:
# article 전체에서 중요한 단어 확인

vectorizer = TfidfVectorizer(
    stop_words = "english",
    max_features = 5000
)

X = vectorizer.fit_transform(sample_df["article"])

In [25]:
feature_names = vectorizer.get_feature_names_out()

In [26]:
def get_top_tfidf_words(row_vector, feature_names, top_n = 10):
    scores = row_vector.toarray().flatten()
    top_indices = scores.argsort()[::-1][:top_n]
    top_words = [feature_names[i] for i in top_indices if scores[i]>0]
    return top_words

In [27]:
top_keywords = []

for i in range(X.shape[0]):
            # 현재 X의 행 개수
    words = get_top_tfidf_words(X[i], feature_names, top_n = 10)
    top_keywords.append(words)

sample_df["article_keywords"] = top_keywords

In [28]:
def keyword_preservation(keywords, summary):
    summary_words = set(summary.lower().split())
    preserved = [word for word in keywords if word in summary_words]

    if len(keywords) == 0:
        return 0
    return len(preserved) / len(keywords)

In [29]:
# TF-IDF Top10 단어들이 highlights에 얼마나 유지되었는지?

sample_df["keyword_preservation"] = sample_df.apply(lambda row:keyword_preservation(
    row["article_keywords"],
    row["highlights"]
    ),
    axis = 1
)

In [30]:
sample_df["keyword_preservation"].describe()

,keyword_preservation
count,5000.00000
mean,0.40030
std,0.19121
min,0.00000
25%,0.30000
50%,0.40000
75%,0.50000
max,1.00000


※ article의 단어들 중 tf-idf가 가장 높은 10개 단어들의 평균 40%가 highlights에서도 그대로 사용됨
- 25%, 50%, 75%를 볼 때 정규분포를 따르는 것처럼 보임

# ■ Highlights의 Article 초-중-후반 반영도 확인

In [35]:
def split_article_parts(article):
    words = article.split()
    n = len(words)

    first = " ".join(words[: n//3])
    middle = " ".join(words[n//3: 2*n//3])
    last = " ".join(words[2*n//3:])

    return first, middle, last

In [41]:
def part_overlap(article_part, summary):
    part_words = set(article_part.lower().split())
    summary_words = set(summary.lower().split())

    if len(summary_words) == 0:
        return 0

    return len(part_words.intersection(summary_words)) / len(summary_words)

In [42]:
first_overlaps = []
middle_overlaps = []
last_overlaps = []

for _, row in sample_df.iterrows():
    first, middle, last = split_article_parts(row["article"])

    first_overlaps.append(part_overlap(first, row["highlights"]))
    middle_overlaps.append(part_overlap(middle, row["highlights"]))
    last_overlaps.append(part_overlap(last, row["highlights"]))

sample_df["first_part_overlap"] = first_overlaps
sample_df["middle_part_overlap"] = middle_overlaps
sample_df["last_part_overlap"] = last_overlaps

In [43]:
sample_df[["first_part_overlap", "middle_part_overlap", "last_part_overlap"]].mean()

,0
first_part_overlap,0.619566
middle_part_overlap,0.475557
last_part_overlap,0.420413


※ first_part 의 오버랩 비율이 60%로 가장 높음.
-  middle은 47%, last는 42%로 끝으로 갈수록 highlights에 반영되는 비율이 낮아짐

In [44]:
high_preservation = sample_df.sort_values(
    "keyword_preservation",
    ascending=False
).head(3)

In [45]:
for _, row in high_preservation.iterrows():
    print("=" * 80)
    print("ARTICLE KEYWORDS:", row["article_keywords"])
    print("\nSUMMARY:")
    print(row["highlights"])
    print("\nARTICLE:")
    print(row["article"][:1000])

ARTICLE KEYWORDS: ['news', 'sites', 'content', 'web', 'free', 'international', 'newspapers', 'access', 'times', 'online']

SUMMARY:
Both newspapers will launch new Web sites in early May, offering free trial to users .
Rupert Murdoch: Free access business model favored by most content providers "flawed"
Financial Times only other major UK newspaper currently charging for online content .
News International hinted plan could be extended to cover company's other UK titles .

ARTICLE:
London, England (CNN) -- News International announced plans Friday to charge for access to The Times and The Sunday Times Web sites starting in June. The publisher said both British newspapers will launch new Web sites in early May and offer a free trial period to registered customers. Starting in June, each site will charge £1 ($1.48) for a day's access or £2 ($2.96) for a week's subscription, News International said. The newspapers are currently available free on a combined site, www.timesonline.co.uk, but

In [46]:
low_preservation = sample_df.sort_values(
    "keyword_preservation",
    ascending=True
).head(3)

In [47]:
for _, row in low_preservation.iterrows():
    print("=" * 80)
    print("ARTICLE KEYWORDS:", row["article_keywords"])
    print("\nSUMMARY:")
    print(row["highlights"])
    print("\nARTICLE:")
    print(row["article"][:1000])

ARTICLE KEYWORDS: ['carolina', 'reported', 'wound', 'maria', 'disappeared', 'rape', 'verdict', 'argument', 'north', 'lee']

SUMMARY:
NEW: Victim's mother says her daughter was the "perfect victim"
The former U.S. Marine was sentenced to life in prison without parole .
He was convicted Monday in the death of a female Marine who was pregnant .
The lawyer for Cesar Laurean says the case will be appealed .

ARTICLE:
(CNN) -- Former U.S. Marine Cesar Laurean was convicted in North Carolina on Monday of first degree murder in the 2007 death of Lance Cpl. Maria Lauterbach, who was eight months pregnant when she died. An autopsy showed that Lauterbach, 20, died of blunt force trauma to the head. Police unearthed her charred body from beneath a barbecue pit in Laurean's backyard in January 2008. She had disappeared the month before. Laurean, who was dressed in black slacks and wore a white shirt and black tie, did not show any emotion as the judge read his sentence of life in prison without par

# ■ Conclusion

- Approximately 80% of summary words also appeared in the original article text, suggesting a strong extractive tendency.
- Around 40% of top TF-IDF keywords from articles were preserved in summaries.
- Summary overlap was highest with the beginning of articles, indicating strong lead bias in news summarization.